# Task A.2.2: Llama 3.1 8B RAG (L-RAG)

**Task ID:** A.2.2
**Date created:** 2026-04-29
**Model:** `meta-llama/Llama-3.1-8B-Instruct` (4-bit quantized)
**Embedding:** `BAAI/bge-m3` (1024-dim, model-independent of generation model)
**Purpose:** Generate RAG-based Brief Hospital Course summaries for the 100-patient VeriFact-BHC cohort, completing the L-RAG corner of the Phase A 2×2 model-method matrix.

This notebook is a **direct port of the RAG v3 (matched-budget, multi-query) condition in `03_rag_pipeline.ipynb`**: only the generation model and the output parquet filenames change. The six clinically-motivated queries, the chunking parameters, the BGE-M3 embeddings, the per-patient FAISS indices, the dedup logic, the retrieval top-k, the 28K retrieval budget, the RAG prompt instruction text, the 31,000-token input budget, and the generation kwargs are preserved exactly so that L-RAG results are directly comparable to M-RAG v3.

**Pairing:** L-RAG is paired with L-ZS (`02b_baseline_llama.ipynb`, Task A.2.1) at matched 31K input budget — the same fairness fix that made M-RAG v3 paired-comparable to M-ZS.

## Step 1: Setup

**Environment:** Google Colab Pro (A100 GPU, High-RAM)
**Data:** VeriFact-BHC processed parquet files (from `01_data_exploration.ipynb`) and Phase A.1 baseline results (for truncation status of each patient).

In [3]:
# Mount Google Drive to access our processed data files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Verify we can see our data files
import os

DATA_DIR = "/content/drive/MyDrive/BMI702 Project/Data"

# List files to confirm the path is correct
print("Files in data directory:")
for f in sorted(os.listdir(DATA_DIR)):
    print(f"  {f}")

Files in data directory:
  L_ZS_BHC_eval_results.parquet
  L_ZS_BHC_results.parquet
  baseline_eval_results.parquet
  baseline_results.parquet
  human_bhcs.parquet
  llm_judge_pairwise_scores.parquet
  llm_judge_pairwise_winner_counts.csv
  llm_judge_plus_auto_metrics_merged.parquet
  llm_judge_qwen3_32b_jsonfix_with_rag3_merged_first5.parquet
  llm_judge_qwen3_32b_jsonfix_with_rag3_pairwise_first5.parquet
  llm_judge_qwen3_32b_jsonfix_with_rag3_pairwise_winner_counts_first5.csv
  llm_judge_qwen3_32b_jsonfix_with_rag3_single_scores_first5.parquet
  llm_judge_qwen3_32b_jsonfix_with_rag3_summary_table_first5.csv
  llm_judge_single_summary_scores.parquet
  llm_judge_strict_with_rag3_merged_first5.parquet
  llm_judge_strict_with_rag3_pairwise_first5.parquet
  llm_judge_strict_with_rag3_pairwise_winner_counts_first5.csv
  llm_judge_strict_with_rag3_single_scores_first5.parquet
  llm_judge_strict_with_rag3_summary_table_first5.csv
  llm_judge_summary_table.csv
  notes.parquet
  proposition_e

### 1.1 Load Processed Data

In [5]:
# Load the processed data + truncation flags from both Phase A.1 (Mistral) and
# Task A.2.1 (Llama) baselines. Per the April 29 SAP amendment, truncation
# populations are model-relative; within-Llama subgroup analysis (L-RAG vs L-ZS)
# uses the L-ZS flag. M-ZS flag is retained for cross-model bookkeeping.
import pandas as pd
import numpy as np

notes = pd.read_parquet(f"{DATA_DIR}/notes.parquet")
human_bhcs = pd.read_parquet(f"{DATA_DIR}/human_bhcs.parquet")

# M-ZS truncation flags (Mistral baseline, Phase A.1)
m_zs_results = pd.read_parquet(f"{DATA_DIR}/baseline_results.parquet")

# L-ZS truncation flags (Llama baseline, Task A.2.1) — primary subgroup label
# for within-Llama analyses, per April 29 SAP amendment.
l_zs_results = pd.read_parquet(f"{DATA_DIR}/L_ZS_BHC_results.parquet")

print(f"Notes: {notes.shape[0]} rows")
print(f"BHC targets: {human_bhcs.shape[0]} patients")
print(f"M-ZS results: {m_zs_results.shape[0]} patients "
      f"({m_zs_results['was_truncated'].sum()} truncated under Mistral)")
print(f"L-ZS results: {l_zs_results.shape[0]} patients "
      f"({l_zs_results['was_truncated'].sum()} truncated under Llama)")

Notes: 4787 rows
BHC targets: 100 patients
M-ZS results: 100 patients (42 truncated under Mistral)
L-ZS results: 100 patients (34 truncated under Llama)


### 1.2 Install Dependencies and Load Tokenizers + Embedding Model

We load **two tokenizers**:

- **Mistral 7B Instruct v0.3 tokenizer** — used **only** to compute chunk boundaries, so the chunks produced here match Phase A.1 byte-for-byte and the per-patient FAISS indices are identical. This is required to honor the "reuse FAISS indices, do not regenerate" rule (Task A.2.2 D6).
- **Llama 3.1 8B Instruct tokenizer** — used for everything else: budget accounting on retrieved chunks (the fairness fix), prompt assembly via the chat template, and final generation.

We also load **BGE-M3** for embeddings. BGE-M3 is independent of the generation model, so the embeddings (and thus FAISS indices) are identical to Phase A.1.

In [6]:
# Install required packages
!pip install -q transformers accelerate bitsandbytes torch sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 96.0 MB/s eta 0:00:00


In [7]:
# Log in to Hugging Face — required for gated Mistral and Llama models
from huggingface_hub import login
login()

In [8]:
# Load the Mistral tokenizer for chunk boundary preservation only
# This produces chunks identical to Phase A.1 (M-RAG v3), so the per-patient FAISS
# indices we build below are byte-for-byte identical to those used by M-RAG v3.
# We never load Mistral model weights — just the tokenizer (~2 MB).
from transformers import AutoTokenizer

chunking_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
print(f"Chunking tokenizer loaded: mistralai/Mistral-7B-Instruct-v0.3 (used only for chunk boundaries)")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Chunking tokenizer loaded: mistralai/Mistral-7B-Instruct-v0.3 (used only for chunk boundaries)


In [9]:
# Load BGE-M3: the same embedding model used in Phase A.1 (M-RAG v3).
# BGE-M3 is independent of the generation model, so embeddings — and therefore
# the per-patient FAISS indices — are identical to Phase A.1.
from sentence_transformers import SentenceTransformer

embed_model_name = "BAAI/bge-m3"
embed_model = SentenceTransformer(embed_model_name)

# Quick sanity check
test_embedding = embed_model.encode(["Patient admitted with chest pain"])
print(f"Embedding model: {embed_model_name}")
print(f"Embedding dimension: {test_embedding.shape[1]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model: BAAI/bge-m3
Embedding dimension: 1024


## Step 2: Chunking Clinical Notes

Break each note into ~300-token chunks with metadata preserved. **Chunking parameters (300-token target, 50-token overlap) are preserved verbatim from Phase A.1 (M-RAG v3).** Boundaries are computed using the Mistral tokenizer to keep chunks byte-identical to those embedded into the Phase A.1 FAISS indices.

In [10]:
# Chunking strategy: split each note into ~300-token pieces (Mistral-tokenized, per Phase A.1)
# We preserve metadata (patient ID, note category, date) with each chunk so the LLM knows
# where retrieved content came from.

def chunk_note(row, tokenizer, target_tokens=300, overlap_tokens=50):
    """
    Split a single clinical note into overlapping chunks.

    Boundaries are computed using the supplied tokenizer. For L-RAG we pass the
    Mistral tokenizer here so chunk text matches Phase A.1 exactly.
    """
    text = row['TEXT']
    tokens = tokenizer.encode(text)

    # If the note is short enough, keep it as one chunk
    if len(tokens) <= target_tokens:
        return [{
            'subject_id': row['SUBJECT_ID'],
            'category': row['CATEGORY'],
            'chartdate': str(row['CHARTDATE']),
            'chunk_text': text,
            'chunk_tokens': len(tokens),
            'note_row_id': row.name
        }]

    # Otherwise, split into overlapping windows
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + target_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)

        chunks.append({
            'subject_id': row['SUBJECT_ID'],
            'category': row['CATEGORY'],
            'chartdate': str(row['CHARTDATE']),
            'chunk_text': chunk_text,
            'chunk_tokens': len(chunk_tokens),
            'note_row_id': row.name
        })

        # Move forward by (target - overlap) so chunks overlap slightly
        start += target_tokens - overlap_tokens

    return chunks

In [11]:
import time

print("Chunking all 4,787 clinical notes (Mistral tokenizer, matching Phase A.1)...")
start = time.time()

all_chunks = []
for _, row in notes.iterrows():
    all_chunks.extend(chunk_note(row, chunking_tokenizer))

chunks_df = pd.DataFrame(all_chunks)
elapsed = time.time() - start

print(f"Done in {elapsed:.1f}s")
print(f"\nChunking summary:")
print(f"  Original notes: {len(notes)}")
print(f"  Total chunks: {len(chunks_df)}")
print(f"  Avg chunks per note: {len(chunks_df) / len(notes):.1f}")
print(f"  Chunk size (Mistral tokens): mean {chunks_df['chunk_tokens'].mean():.0f}, "
      f"median {chunks_df['chunk_tokens'].median():.0f}")

# How many chunks per patient?
per_patient = chunks_df.groupby('subject_id').size()
print(f"\nChunks per patient: mean {per_patient.mean():.0f}, median {per_patient.median():.0f}, "
      f"range {per_patient.min()}-{per_patient.max()}")

Chunking all 4,787 clinical notes (Mistral tokenizer, matching Phase A.1)...
Done in 12.6s

Chunking summary:
  Original notes: 4787
  Total chunks: 23663
  Avg chunks per note: 4.9
  Chunk size (Mistral tokens): mean 266, median 300

Chunks per patient: mean 237, median 110, range 33-1561


## Step 3: Embed Chunks and Build FAISS Indices

Convert each chunk into a 1024-dim BGE-M3 vector, then build a per-patient FAISS index for fast similarity search. Embeddings and indices are identical to Phase A.1 (Task A.2.2 D6).

In [12]:
# Embed all chunks with BGE-M3
import faiss

print(f"Embedding {len(chunks_df)} chunks with BGE-M3...")
start = time.time()

all_texts = chunks_df['chunk_text'].tolist()

# Encode in batches of 256 to avoid OOM
batch_size = 256
all_embeddings = []

for i in range(0, len(all_texts), batch_size):
    batch = all_texts[i:i + batch_size]
    batch_emb = embed_model.encode(batch, show_progress_bar=False)
    all_embeddings.append(batch_emb)

    if (i // batch_size + 1) % 10 == 0:
        print(f"  Encoded {min(i + batch_size, len(all_texts))}/{len(all_texts)} chunks")

all_embeddings = np.vstack(all_embeddings).astype('float32')

elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s")
print(f"Embeddings shape: {all_embeddings.shape}")

Embedding 23663 chunks with BGE-M3...
  Encoded 2560/23663 chunks
  Encoded 5120/23663 chunks
  Encoded 7680/23663 chunks
  Encoded 10240/23663 chunks
  Encoded 12800/23663 chunks
  Encoded 15360/23663 chunks
  Encoded 17920/23663 chunks
  Encoded 20480/23663 chunks
  Encoded 23040/23663 chunks

Done in 220.5s
Embeddings shape: (23663, 1024)


In [13]:
# Build a FAISS index for each patient (cosine similarity via IndexFlatIP on L2-normalized vectors)
patient_ids = chunks_df['subject_id'].unique()
patient_indices = {}    # {patient_id: faiss_index}
patient_chunks = {}     # {patient_id: dataframe of chunks}
patient_embeddings = {} # {patient_id: numpy array of embeddings}

print(f"Building FAISS indices for {len(patient_ids)} patients...")

for pid in patient_ids:
    mask = chunks_df['subject_id'] == pid
    p_chunks = chunks_df[mask].reset_index(drop=True)
    p_emb = all_embeddings[mask.values]

    # Normalize for cosine similarity
    faiss.normalize_L2(p_emb)

    index = faiss.IndexFlatIP(p_emb.shape[1])
    index.add(p_emb)

    patient_indices[pid] = index
    patient_chunks[pid] = p_chunks
    patient_embeddings[pid] = p_emb

print(f"Done! Built {len(patient_indices)} patient indices")

Building FAISS indices for 100 patients...
Done! Built 100 patient indices


## Step 4: Load Llama 3.1 8B and Recompute Chunk Tokens for Budget Fairness

Now load the generation model (Llama 3.1 8B Instruct, 4-bit quantized — same `BitsAndBytesConfig` as Task A.2.1).

**Critical fairness step (Task A.2.2 D9):** the source M-RAG v3 budget enforcement uses `chunk_tokens` (Mistral-tokenized) summed against a hardcoded `max_total_tokens=28000`. For L-RAG, we **recompute `chunk_tokens` under the Llama tokenizer** so the same 28K budget is enforced in Llama tokens — preserving the upstream logic ("RAG context fits in the 31K input budget with prompt-template headroom") while honoring the model swap. The chunk *text* is unchanged; only the per-chunk token count is recomputed.

In [14]:
# Load Llama 3.1 8B Instruct with the same 4-bit quantization config as Task A.2.1.
# NF4 + fp16 compute + double-quant.
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "meta-llama/Llama-3.1-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print(f"Model loaded: {model_name}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded: meta-llama/Llama-3.1-8B-Instruct
GPU memory used: 8.0 GB


In [15]:
# Recompute chunk_tokens under the Llama tokenizer (fairness fix, Task A.2.2 D9).
# Chunk text is unchanged — only the per-chunk token count is updated so that the
# 28K retrieval budget is enforced in Llama tokens.
print("Recomputing chunk_tokens under Llama tokenizer (fairness fix)...")
start = time.time()

# Mistral-tokenized counts (Phase A.1 originals) — kept for comparison only
chunks_df['chunk_tokens_mistral'] = chunks_df['chunk_tokens']

# Overwrite chunk_tokens with Llama-tokenized counts. Budget enforcement below
# operates on this column, so the 28K cap is now Llama-fair.
chunks_df['chunk_tokens'] = chunks_df['chunk_text'].apply(
    lambda t: len(tokenizer.encode(t, add_special_tokens=False))
)

# Propagate the recomputed counts into the per-patient chunk frames used by retrieval
for pid in patient_ids:
    p = patient_chunks[pid]
    # Re-pull the matching rows from the updated chunks_df preserving order
    refreshed = chunks_df[chunks_df['subject_id'] == pid].reset_index(drop=True)
    patient_chunks[pid] = refreshed

elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s")
print(f"\nChunk token counts (per chunk):")
print(f"  Mistral (Phase A.1): mean {chunks_df['chunk_tokens_mistral'].mean():.1f}, "
      f"median {chunks_df['chunk_tokens_mistral'].median():.0f}")
print(f"  Llama (this notebook): mean {chunks_df['chunk_tokens'].mean():.1f}, "
      f"median {chunks_df['chunk_tokens'].median():.0f}")

Recomputing chunk_tokens under Llama tokenizer (fairness fix)...
Done in 12.3s

Chunk token counts (per chunk):
  Mistral (Phase A.1): mean 266.3, median 300
  Llama (this notebook): mean 219.0, median 245


## Step 5: Multi-Query Retrieval (RAG v3)

Six clinically-motivated queries, one per BHC component, retrieved independently then deduplicated. **Queries are preserved verbatim from M-RAG v3 (Task A.2.2 D7).** Top-k, dedup logic, and `max_total_tokens=28000` are preserved from M-RAG v3 — the budget value is unchanged but is now applied to Llama-tokenized chunk counts (fairness fix above).

In [16]:
# Six clinically-motivated queries — preserved verbatim from M-RAG v3
# (Task A.2.2 D7). Each targets a different aspect of the hospital course,
# mirroring how a physician mentally organizes a BHC.

BHC_QUERIES = {
    'admission': "Reason for admission, chief complaint, presenting symptoms, emergency department evaluation",
    'diagnoses': "Primary diagnosis, secondary diagnoses, differential diagnosis, assessment",
    'workup': "Laboratory results, imaging findings, diagnostic test results, pathology",
    'treatment': "Medications started, surgical procedures, interventions performed, treatments administered",
    'course': "Clinical progression, complications, response to treatment, significant events during hospitalization",
    'discharge': "Condition at discharge, discharge disposition, follow-up plan, discharge medications"
}

In [17]:
# RAG v3 retrieval: multi-query, deduplicated, matched-budget (~28K tokens).
# Identical to retrieve_multi_query_v3 in 03_rag_pipeline.ipynb. The only
# behavioral difference is that chunk_tokens is now Llama-tokenized, so the
# 28000 budget is enforced in Llama tokens.

def retrieve_multi_query_v3(patient_id, queries, embed_model, patient_indices,
                            patient_chunks, chunks_per_query=20, max_total_tokens=28000):
    """
    Multi-query retrieval with a 28K token budget (Llama tokens for L-RAG).

    Args:
        queries: Dict of {section_name: query_text} — the six BHC queries above.
        chunks_per_query: How many chunks per query before deduplication.
        max_total_tokens: Total retrieval budget in Llama tokens (recomputed
            from Mistral; original M-RAG v3 value 28000 in Mistral tokens —
            same constant, applied against Llama-tokenized chunk_tokens for
            paired-comparable budget enforcement with L-ZS at 31K).
    """
    all_selected = []
    seen_fingerprints = set()

    for section, query in queries.items():
        query_emb = embed_model.encode([query]).astype('float32')
        faiss.normalize_L2(query_emb)

        p_chunks = patient_chunks[patient_id]
        actual_fetch = min(chunks_per_query * 3, patient_indices[patient_id].ntotal)
        scores, indices = patient_indices[patient_id].search(query_emb, actual_fetch)

        section_count = 0
        for idx, score in zip(indices[0], scores[0]):
            if section_count >= chunks_per_query:
                break

            chunk = p_chunks.iloc[idx]
            text = chunk['chunk_text']
            mid = len(text) // 2
            fingerprint = text[max(0, mid-100):mid+100]

            is_dup = False
            for seen in seen_fingerprints:
                overlap = len(set(fingerprint.split()) & set(seen.split()))
                total = max(len(set(fingerprint.split())), 1)
                if overlap / total > 0.6:
                    is_dup = True
                    break

            if not is_dup:
                all_selected.append({
                    **chunk.to_dict(),
                    'similarity_score': score,
                    'query_section': section
                })
                seen_fingerprints.add(fingerprint)
                section_count += 1

    retrieved = pd.DataFrame(all_selected)

    # Enforce token budget — keep highest-scoring chunks until cumulative
    # Llama-tokenized chunk_tokens exceeds 28K, then sort chronologically.
    retrieved = retrieved.sort_values('similarity_score', ascending=False)
    cumulative_tokens = retrieved['chunk_tokens'].cumsum()
    retrieved = retrieved[cumulative_tokens <= max_total_tokens]
    retrieved = retrieved.sort_values('chartdate')
    total_tokens = retrieved['chunk_tokens'].sum()

    return retrieved, total_tokens

In [18]:
# Quick sanity check on the same patient used in 03_rag_pipeline.ipynb (1084)
test_pid = human_bhcs['subject_id'].iloc[0]

retrieved, retrieved_tokens = retrieve_multi_query_v3(
    test_pid, BHC_QUERIES, embed_model, patient_indices, patient_chunks
)

print(f"Patient {test_pid}: {len(retrieved)} chunks, {retrieved_tokens} Llama tokens retrieved")
print(f"\nChunks per query section:")
print(retrieved['query_section'].value_counts().to_string())
print(f"\nNote categories retrieved:")
print(retrieved['category'].value_counts().to_string())

Patient 1084: 44 chunks, 9784 Llama tokens retrieved

Chunks per query section:
query_section
admission    20
diagnoses    19
treatment     3
workup        2

Note categories retrieved:
category
Physician      33
Nursing         8
Respiratory     2
ECG             1


## Step 6: RAG Generation with Llama 3.1 8B

Feed retrieved chunks to Llama 3.1 8B with the same prompt instruction text used in M-RAG v3 (Task A.2.2 D8: lifted byte-for-byte into `messages[0]['content']`). Generation kwargs match Task A.2.1 (L-ZS).

In [19]:
# RAG prompt instruction text — lifted byte-for-byte from 03_rag_pipeline.ipynb.
# We do NOT rewrite, restructure, paraphrase, or "Llama-ify" this content. Only
# the chat-template wrapping changes (Mistral [INST]...[/INST] -> Llama chat
# template via apply_chat_template). This is a model-swap experiment, not a
# prompt-swap experiment.

RAG_PROMPT = """You are a physician. Based on the following relevant excerpts from a patient's clinical notes during their hospital stay, write a Brief Hospital Course summarizing the key events, findings, treatments, and outcomes.

These excerpts have been selected as the most relevant portions from the patient's full medical record. They are presented in chronological order.

Relevant Clinical Excerpts:
{context}

Brief Hospital Course:"""


def build_rag_context(retrieved_df):
    """Format retrieved chunks into a context string. Each chunk gets a
    [Category — Date] header so the LLM knows the source of each excerpt."""
    sections = []
    for _, chunk in retrieved_df.iterrows():
        header = f"[{chunk['category']} — {chunk['chartdate']}]"
        sections.append(f"{header}\n{chunk['chunk_text']}")
    return "\n\n".join(sections)


def generate_bhc_rag(retrieved_df, model, tokenizer, max_new_tokens=1024):
    """
    Generate a BHC from retrieved chunks using Llama 3.1 8B.

    Single user-message chat-template wrapping, no system role (Task A.2.1 D1:
    Option A — preserves the M-ZS / M-RAG v3 structure where "You are a
    physician..." is part of the user content, since Mistral v0.3 has no system
    role). apply_chat_template handles the model-specific format.

    Generation kwargs (temperature, top_p, do_sample, repetition_penalty) match
    Task A.2.1 / Task A.1 exactly.
    """
    context = build_rag_context(retrieved_df)
    full_prompt = RAG_PROMPT.format(context=context)

    messages = [{"role": "user", "content": full_prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_tokens = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1
        )

    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return generated_text, input_tokens

In [20]:
# Test on patient 1084 before running the full cohort
print(f"Generating L-RAG BHC for patient {test_pid}...")
print(f"Retrieved context: {retrieved_tokens} Llama tokens "
      f"(vs full notes: {notes[notes['SUBJECT_ID'] == test_pid]['TEXT'].apply(lambda x: len(tokenizer.encode(x))).sum()} tokens)")

generated_rag, input_tokens = generate_bhc_rag(retrieved, model, tokenizer)

print(f"Input tokens (prompt + context): {input_tokens}")
print(f"Generated BHC ({len(tokenizer.encode(generated_rag))} tokens):")
print("=" * 80)
print(generated_rag)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generating L-RAG BHC for patient 1084...
Retrieved context: 9784 Llama tokens (vs full notes: 13238 tokens)
Input tokens (prompt + context): 10443
Generated BHC (523 tokens):
**Patient Information**

* Name: [**Name10 (NameIs)**]
* Age: 61 years old
* Sex: Male
* Chief Complaint: Altered Mental Status (AMS)

**Admission**

* Date: August 2, 2198
* Time: Unknown
* Mode of Arrival: Emergency Department (ED)
* Symptoms: Construction workers found the patient "acting funny" and naked at home; reported confusion and possibly dilated pupils

**Initial Assessment**

* Initial vital signs: 97/174 mmHg, 67 bpm, 16 mmHg, 100% oxygen saturation
* Initial laboratory results: serum acetaminophen, ethanol, benzodiazepines, barbiturates, tricyclic antidepressants negative; urine barbiturates positive; urine opiates positive; urine methadone positive
* Initial imaging: head CT scan showed no acute process; lumbar puncture (LP) was pending

**Treatment**

* Administration of naloxone (Narcan) by emerge

## Step 7: Run L-RAG on All 100 Patients

Generate L-RAG BHCs for all 100 patients. We track retrieved tokens, input tokens, generation time, and the M-ZS truncation flag (carried from Phase A.1 baseline) so subgroup analyses in Task A.2.7 / A.2.8 can split by Mistral-baseline truncation status.

In [21]:
# Smoke-test switch: set LIMIT_PATIENTS = 5 (or any small N) to run on the first
# N patients before committing to the full cohort. Leave as None for the real run.
# Convention shared with 02b_baseline_llama.ipynb (Task A.2.1 D2).
LIMIT_PATIENTS = None  # set to 5 for smoke test

In [22]:
# Run RAG v3 on all patients
rag_results = []

patients_to_run = human_bhcs.head(LIMIT_PATIENTS) if LIMIT_PATIENTS else human_bhcs

print(f"Running L-RAG on {len(patients_to_run)} patients...")
print("-" * 60)

start_total = time.time()

for i, row in patients_to_run.iterrows():
    patient_id = row['subject_id']

    # Step 1: Retrieve relevant chunks (multi-query, dedup, 28K Llama-token budget)
    retrieved_df, retrieved_tokens = retrieve_multi_query_v3(
        patient_id, BHC_QUERIES, embed_model, patient_indices, patient_chunks
    )

    # Step 2: Generate BHC from retrieved chunks
    start = time.time()
    generated, input_tokens = generate_bhc_rag(retrieved_df, model, tokenizer)
    elapsed = time.time() - start

    # Total Llama tokens in the patient's full notes (for compression-ratio reporting)
    full_tokens = notes[notes['SUBJECT_ID'] == patient_id]['TEXT'].apply(
        lambda x: len(tokenizer.encode(x))
    ).sum()

    # Truncation flags from both baselines:
    # - was_truncated_m_zs: Mistral baseline (Phase A.1) — cross-model bookkeeping.
    # - was_truncated_l_zs: Llama baseline (Task A.2.1) — primary subgroup label
    #   for within-Llama analyses (L-RAG vs L-ZS), per April 29 SAP amendment.
    was_truncated_m_zs = m_zs_results[
        m_zs_results['subject_id'] == patient_id
    ]['was_truncated'].iloc[0]
    was_truncated_l_zs = l_zs_results[
        l_zs_results['subject_id'] == patient_id
    ]['was_truncated'].iloc[0]

    rag_results.append({
        'subject_id': patient_id,
        'full_note_tokens': full_tokens,
        'retrieved_tokens': retrieved_tokens,
        'input_tokens': input_tokens,
        'compression_ratio': retrieved_tokens / full_tokens,
        'was_truncated_m_zs': was_truncated_m_zs,
        'was_truncated_l_zs': was_truncated_l_zs,
        'generated_bhc': generated,
        'generated_tokens': len(tokenizer.encode(generated)),
        'generation_time_sec': round(elapsed, 1),
        'human_bhc': row['brief_hospital_course'],
        'n_chunks_retrieved': len(retrieved_df)
    })

    if len(rag_results) % 10 == 0:
        print(f"  {len(rag_results)}/{len(patients_to_run)} | Last: {full_tokens} full -> "
              f"{retrieved_tokens} retrieved ({retrieved_tokens/full_tokens:.0%}) | "
              f"{elapsed:.1f}s | L-ZS truncated: {was_truncated_l_zs}")

total_time = time.time() - start_total
print("-" * 60)
print(f"Done! Total time: {total_time/60:.1f} minutes")

rag_results_df = pd.DataFrame(rag_results)

Running L-RAG on 100 patients...
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  10/100 | Last: 78219 full -> 21116 retrieved (27%) | 44.9s | L-ZS truncated: True


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  20/100 | Last: 159768 full -> 27621 retrieved (17%) | 41.4s | L-ZS truncated: True


...

  80/100 | Last: 18851 full -> 13950 retrieved (74%) | 38.8s | L-ZS truncated: False


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  90/100 | Last: 18368 full -> 15954 retrieved (87%) | 45.8s | L-ZS truncated: False


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  100/100 | Last: 22243 full -> 11998 retrieved (54%) | 45.4s | L-ZS truncated: False
------------------------------------------------------------
Done! Total time: 76.2 minutes


In [23]:
# Save L-RAG generation results to Drive — schema column-for-column matches
# L_ZS_BHC_results.parquet (plus RAG-specific columns for retrieval bookkeeping).
rag_results_df.to_parquet(f"{DATA_DIR}/L_RAG_BHC_results.parquet", index=False)
print(f"Saved {len(rag_results_df)} L-RAG results to Drive")

Saved 100 L-RAG results to Drive


## Step 8: Reference Evaluation — ROUGE + BERTScore

Standard automated metrics against the human-written gold standard, matching Task A.2.1 (L-ZS) so the L-RAG vs L-ZS comparison is apples-to-apples. **Proposition-level fact recall and PDSQI-9 LLM-as-judge are deferred to Task A.2.7 (shared evaluation notebook) — not bundled here.**

In [24]:
!pip install -q rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.1 MB/s eta 0:00:00


In [25]:
from rouge_score import rouge_scorer

# Same ROUGE config as Task A.2.1 / Task A.1
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rag_rouge = []
for _, row in rag_results_df.iterrows():
    scores = scorer.score(row['human_bhc'], row['generated_bhc'])
    rag_rouge.append({
        'subject_id': row['subject_id'],
        'rouge1_f': scores['rouge1'].fmeasure,
        'rouge2_f': scores['rouge2'].fmeasure,
        'rougeL_f': scores['rougeL'].fmeasure,
    })

rag_rouge_df = pd.DataFrame(rag_rouge)

print(f"L-RAG ROUGE Scores (n={len(rag_rouge_df)}):")
print(f"  ROUGE-1 F1: {rag_rouge_df['rouge1_f'].mean():.3f} (±{rag_rouge_df['rouge1_f'].std():.3f})")
print(f"  ROUGE-2 F1: {rag_rouge_df['rouge2_f'].mean():.3f} (±{rag_rouge_df['rouge2_f'].std():.3f})")
print(f"  ROUGE-L F1: {rag_rouge_df['rougeL_f'].mean():.3f} (±{rag_rouge_df['rougeL_f'].std():.3f})")

L-RAG ROUGE Scores (n=100):
  ROUGE-1 F1: 0.340 (±0.063)
  ROUGE-2 F1: 0.080 (±0.034)
  ROUGE-L F1: 0.159 (±0.030)


In [26]:
from bert_score import score as bert_score_fn

print("Computing BERTScore (this may take a few minutes)...")

P, R, F1 = bert_score_fn(
    rag_results_df['generated_bhc'].tolist(),
    rag_results_df['human_bhc'].tolist(),
    lang='en',
    verbose=True
)

rag_results_df['bertscore_p'] = P.numpy()
rag_results_df['bertscore_r'] = R.numpy()
rag_results_df['bertscore_f1'] = F1.numpy()

print(f"\nL-RAG BERTScore (n={len(rag_results_df)}):")
print(f"  Precision: {rag_results_df['bertscore_p'].mean():.3f} (±{rag_results_df['bertscore_p'].std():.3f})")
print(f"  Recall:    {rag_results_df['bertscore_r'].mean():.3f} (±{rag_results_df['bertscore_r'].std():.3f})")
print(f"  F1:        {rag_results_df['bertscore_f1'].mean():.3f} (±{rag_results_df['bertscore_f1'].std():.3f})")

Computing BERTScore (this may take a few minutes)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 3.36 seconds, 29.72 sentences/sec

L-RAG BERTScore (n=100):
  Precision: 0.819 (±0.014)
  Recall:    0.810 (±0.014)
  F1:        0.814 (±0.011)


In [27]:
# Merge ROUGE into the eval frame and save
rag_eval = rag_results_df.merge(rag_rouge_df, on='subject_id')
rag_eval.to_parquet(f"{DATA_DIR}/L_RAG_BHC_eval_results.parquet", index=False)
print(f"Saved L-RAG evaluation results to Drive")

# Quick summary
print(f"""
Model: Llama 3.1 8B Instruct (4-bit quantized)
Patients: {len(rag_results_df)} (VeriFact-BHC cohort)
Retrieval budget: 28K Llama tokens (paired with L-ZS 31K input budget)
Six BHC queries (verbatim from M-RAG v3)

ROUGE-1 F1: {rag_rouge_df['rouge1_f'].mean():.3f}
ROUGE-2 F1: {rag_rouge_df['rouge2_f'].mean():.3f}
ROUGE-L F1: {rag_rouge_df['rougeL_f'].mean():.3f}
BERTScore F1: {rag_results_df['bertscore_f1'].mean():.3f}

Generated BHC length: mean {rag_results_df['generated_tokens'].mean():.0f} tokens
Retrieved context: mean {rag_results_df['retrieved_tokens'].mean():.0f} Llama tokens
""")

Saved L-RAG evaluation results to Drive

Model: Llama 3.1 8B Instruct (4-bit quantized)
Patients: 100 (VeriFact-BHC cohort)
Retrieval budget: 28K Llama tokens (paired with L-ZS 31K input budget)
Six BHC queries (verbatim from M-RAG v3)

ROUGE-1 F1: 0.340
ROUGE-2 F1: 0.080
ROUGE-L F1: 0.159
BERTScore F1: 0.814

Generated BHC length: mean 536 tokens
Retrieved context: mean 15253 Llama tokens



## Port-decision log

**Source notebook:** `notebooks/03_rag_pipeline.ipynb` (Task A.1, M-RAG v3 condition)
**Target notebook:** `notebooks/03b_rag_llama.ipynb` (Task A.2.2, L-RAG)
**Port date:** 2026-04-29
**Model swap:** `mistralai/Mistral-7B-Instruct-v0.3` → `meta-llama/Llama-3.1-8B-Instruct`

### Decisions carried from A.2.1

- **D1 — System role:** Option A (no Llama system role; single user-message chat-template wrapper). The original Mistral v0.3 has no system role, so "You are a physician..." was embedded inside the user content. We preserve that structure exactly so the L-RAG vs M-RAG v3 comparison varies only the model.
- **D2 — `LIMIT_PATIENTS` smoke-test convention:** module-level `LIMIT_PATIENTS = None` for the full cohort, set to `5` for smoke test. Convention shared with `02b_baseline_llama.ipynb`.
- **D3 — Colab-only execution:** bitsandbytes 4-bit quantization is unsupported on Apple Silicon MPS; this notebook is intended for Colab A100.
- **D4 — 4-bit quantization config:** `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")` — identical to A.2.1.
- **D5 — Generation kwargs:** `temperature=0.1, top_p=0.9, do_sample=True, repetition_penalty=1.1, max_new_tokens=1024` — identical to A.2.1 / A.1.

### Decisions new to A.2.2

- **D6 — FAISS indices reused, not regenerated.** Per-patient FAISS indices are byte-identical to Phase A.1 because: (a) BGE-M3 embeddings are independent of the generation model, and (b) we use the **Mistral tokenizer** for chunk-boundary computation (`chunking_tokenizer` in Step 1.2), so chunk text matches Phase A.1 exactly. The Llama tokenizer is loaded only for budget accounting, prompt assembly, and generation.
- **D7 — Six clinically-motivated queries preserved verbatim** from M-RAG v3 (admission / diagnoses / workup / treatment / course / discharge). No paraphrasing, no Llama-ification.
- **D8 — RAG prompt instruction text lifted byte-for-byte** into `messages[0]["content"]`. Only the chat-template wrapping changes (Mistral `[INST]...[/INST]` → Llama chat-template via `apply_chat_template`). The instruction content does not.
- **D9 — Budget calculation:** **recomputed** under the Llama tokenizer.
  - Source M-RAG v3 used `max_total_tokens=28000` as a hardcoded literal, applied against `chunk_tokens` that were Mistral-tokenized at chunking time. This is exactly the "hardcoded Mistral-derived constant" pattern flagged in the A.2.2 plan.
  - **Original M-RAG v3 value:** `max_total_tokens=28000` enforced against Mistral-tokenized chunk counts.
  - **Recomputed L-RAG v3 value:** `max_total_tokens=28000` enforced against **Llama-tokenized** `chunk_tokens` (`chunks_df['chunk_tokens']` is overwritten in Step 4 using `tokenizer.encode(chunk_text, add_special_tokens=False)` where `tokenizer` is Llama; original Mistral counts retained for reference in `chunks_df['chunk_tokens_mistral']`).
  - **Upstream logic preserved:** "RAG retrieval budget = generation context budget − prompt-template overhead". Phase A.1 chose 28K to leave ~3K headroom under the 31K Mistral-token input budget. We hold the same 28K cap and the same 31K input budget under the Llama tokenizer (matching A.2.1 L-ZS), so the headroom logic is preserved exactly. The constant value is unchanged but the unit (token count) is now Llama-fair, which is the fairness fix that makes L-RAG paired-comparable to L-ZS.

- **D10 — Truncation flags carried forward from both baselines.** Phase A.1 source notebook recorded `was_truncated_baseline` (M-ZS Mistral). Per April 29 SAP amendment defining truncation populations as model-relative, the L-RAG output schema now records both `was_truncated_m_zs` (Mistral baseline, cross-model bookkeeping) and `was_truncated_l_zs` (Llama baseline, primary subgroup label for within-Llama L-RAG vs L-ZS analyses). Both are loaded in Step 1.1 from `baseline_results.parquet` and `L_ZS_BHC_results.parquet` respectively.
